# PharmaRL — GRPO training on the drug-discovery env

**What this does**: trains Llama-3.2-1B (LoRA + 4-bit) to design small-molecule drug candidates by editing SMILES strings. Talks to a deployed PharmaRL OpenEnv (HF Space) over HTTP. Logs reward curves to W&B.

**Runtime**: Colab T4 (free) is enough thanks to Unsloth 4-bit + LoRA.

**Before you run**: edit the `ENV_URL` constant in the Config cell to point at your deployed env Space.

## 1. Clone repo

In [ ]:
# The notebook imports from server/, models, and scripts.train_grpo, so the
# repo must live on disk. Re-running this cell is safe (idempotent).
import os, subprocess, sys
REPO_DIR = '/content/pharmarl'
if not os.path.exists(REPO_DIR):
    subprocess.check_call(['git', 'clone',
                           'https://github.com/AnshumanAtrey/pharmarl.git',
                           REPO_DIR])
else:
    subprocess.check_call(['git', '-C', REPO_DIR, 'pull', '--ff-only'])
# Optional: switch to the branch that has the T4 fixes — comment out for main.
subprocess.check_call(['git', '-C', REPO_DIR, 'checkout', 'temp-train-grpo-t4-fix'])
subprocess.check_call(['git', '-C', REPO_DIR, 'pull', '--ff-only'])
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print('repo at:', REPO_DIR, ' branch:',
      subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', '--abbrev-ref', 'HEAD']).decode().strip())

## 2. Install deps

In [ ]:
!pip install -q unsloth
!pip install -q --upgrade --no-cache-dir 'trl>=0.11.0' 'transformers>=4.40.0' peft accelerate bitsandbytes
!pip install -q openenv-core selfies rdkit-pypi PyTDC wandb huggingface_hub

## 3. Config

In [ ]:
import os

ENV_URL = os.environ.get('PHARMARL_ENV_URL', 'http://localhost:8000')  # set to HF Space URL once deployed
MODEL_NAME = 'unsloth/Llama-3.2-1B-Instruct'
MAX_SEQ_LEN = 1024
LORA_RANK = 16
NUM_GENERATIONS = 4           # G in GRPO — must be >= 2; T4 fits 4 with 1B
MAX_TRAINING_STEPS = 200      # realistic on T4 in 8h is 150-300 steps
SFT_WARMUP_STEPS = 120        # format-prime before GRPO
LR = 5e-6
WANDB_PROJECT = 'pharmarl'
WANDB_RUN_NAME = 'colab-multitarget-heldout'

# Held-out generalization test setup:
#   Train: rotate per-episode through TRAINING_TARGETS
#   Eval:  measure on HELD_OUT_TARGET — a target the model NEVER sees during training.
TRAINING_TARGETS = ['DRD2', 'GSK3B']
HELD_OUT_TARGET = 'JNK3'
EVAL_EPISODES_PER_TARGET = 8
EVAL_DIFFICULTY = 'easy'      # binding component is active here (trivial is QED-only)

## 4. Load model with Unsloth (4-bit + LoRA)

In [ ]:
from unsloth import FastLanguageModel
import torch

# Hardware-aware setup. T4 (no bf16) → fp16 + 4-bit; Ampere+ → bf16 base.
bf16_ok = torch.cuda.is_bf16_supported()
load_kwargs = (
    dict(load_in_4bit=False, dtype=torch.bfloat16) if bf16_ok
    else dict(load_in_4bit=True, dtype=torch.float16)
)
print('hw bf16_supported=', bf16_ok, ' →', load_kwargs)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    fast_inference=False,  # vLLM is heavy and unnecessary; ~2x slower rollouts but installs cleanly on Kaggle/Colab
    **load_kwargs,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=LORA_RANK,
    use_gradient_checkpointing=True,  # stock — bypasses unsloth fast_lora kernel (avoids T4 NaN)
    random_state=42,
)
# Stock GC needs gradients to flow through frozen embeddings.
if hasattr(model, 'enable_input_require_grads'):
    model.enable_input_require_grads()

# Sanity: dump trainable param dtype histogram.
hist = {}
n = 0
for p in model.parameters():
    if p.requires_grad:
        n += 1
        hist[str(p.dtype)] = hist.get(str(p.dtype), 0) + 1
print(f'{n} trainable params; dtypes={hist}')

## 5. Connect to env

In [ ]:
import requests
import uuid

# Smoke test: hit the env once. NOTE: episode_id required — the env keeps state across calls per id.
_eid = str(uuid.uuid4())
r = requests.post(f'{ENV_URL}/reset', json={'difficulty': 'trivial', 'episode_id': _eid}).json()
print('reset OK:', r['observation']['smiles'], '   episode_id:', r['observation']['episode_id'])
print('vocab:', r['observation']['available_fragments'])

## 6. Rollout helper — interact with env to produce trajectories

In [ ]:
import re
import json as _json

_JSON_RE = re.compile(r'\{[^{}]*\}')

def parse_action(text: str):
    for m in _JSON_RE.findall(text):
        try:
            return _json.loads(m)
        except Exception:
            continue
    return None


SYSTEM = '''You are a medicinal chemist designing a small-molecule drug. Each turn you edit a molecule by issuing a single JSON action.

Output format (respond with EXACTLY ONE JSON object on its own line — no prose, no markdown fences):
  {"action_type": "ADD_FRAGMENT", "fragment": "<smiles>", "position": 0}
  {"action_type": "REMOVE_FRAGMENT", "position": 0}
  {"action_type": "SUBSTITUTE_ATOM", "position": 0, "new_atom": "F"}
  {"action_type": "TERMINATE"}'''


def rollout_episode(model, tokenizer, env_url, difficulty='easy', target=None,
                    max_new_tokens=80, max_steps=20):
    """Multi-step rollout using Llama's chat template (gets reliable JSON parse rate).

    `target` selects the binding oracle: 'DRD2' / 'GSK3B' / 'JNK3' / None for default.
    """
    episode_id = str(uuid.uuid4())
    body = {'difficulty': difficulty, 'episode_id': episode_id}
    if target is not None:
        body['target'] = target
    obs = requests.post(f'{env_url}/reset', json=body).json()['observation']
    actions, rewards = [], []
    cumulative = 0.0
    for _ in range(max_steps):
        user_msg = (
            f'Target: {obs.get("target", "default")}\n'
            f'SMILES: {obs["smiles"]}\n'
            f'Fragments: {obs["available_fragments"][:8]}\n'
            f'Valid actions: {obs["valid_actions"]}'
        )
        messages = [
            {'role': 'system', 'content': SYSTEM},
            {'role': 'user', 'content': user_msg},
        ]
        prompt_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
        )
        inputs = tokenizer(prompt_text, return_tensors='pt').to(model.device)
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens, temperature=0.7,
            do_sample=True, pad_token_id=tokenizer.eos_token_id,
        )
        txt = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        action = parse_action(txt) or {'action_type': 'ADD_FRAGMENT', 'fragment': 'C', 'position': 0}
        step = requests.post(
            f'{env_url}/step',
            json={'action': action, 'episode_id': episode_id},
        ).json()
        actions.append(action)
        rewards.append(step['reward'])
        cumulative += step['reward']
        obs = step['observation']
        if step['done']:
            break
    return {
        'actions': actions,
        'rewards': rewards,
        'cumulative': cumulative,
        'final_smiles': obs['smiles'],
        'episode_id': episode_id,
        'target': target,
    }

## 7. Smoke run (5 episodes BEFORE training)

In [ ]:
import statistics

FastLanguageModel.for_inference(model)

# (1) Sanity smoke on the easy tier with a training target — proves the loop works
smoke = [rollout_episode(model, tokenizer, ENV_URL, EVAL_DIFFICULTY, TRAINING_TARGETS[0]) for _ in range(3)]
print(f'Smoke (DRD2, easy): mean cum = {statistics.mean(r["cumulative"] for r in smoke):.3f}')
for i, r in enumerate(smoke):
    print(f'  ep{i+1}: cum={r["cumulative"]:.3f}  final={r["final_smiles"]}')

# (2) HELD-OUT baseline — untrained model on JNK3 (the target it will NEVER see during training).
print(f'\nUntrained baseline on HELD-OUT target {HELD_OUT_TARGET}:')
untrained_held_out = [rollout_episode(model, tokenizer, ENV_URL, EVAL_DIFFICULTY, HELD_OUT_TARGET)
                      for _ in range(EVAL_EPISODES_PER_TARGET)]
untrained_mean = statistics.mean(r['cumulative'] for r in untrained_held_out)
print(f'  Untrained mean cumulative on {HELD_OUT_TARGET}: {untrained_mean:+.3f}')
for i, r in enumerate(untrained_held_out):
    print(f'  ep{i+1}: cum={r["cumulative"]:+.3f}  final={r["final_smiles"]}')

## 8. SFT format priming + GRPO training

This delegates to the validated training functions in `scripts/train_grpo.py` so the notebook stays in sync with the script. SFT primes JSON output format (~5 min); GRPO does the actual reward optimization (~3-6h on T4 for 200 steps).

In [ ]:
import os
os.environ.setdefault('WANDB_PROJECT', WANDB_PROJECT)
os.environ.setdefault('WANDB_RUN_NAME', WANDB_RUN_NAME)
# Set this if you want W&B logging (otherwise training runs without W&B):
# os.environ['WANDB_API_KEY'] = 'your-key-here'

from scripts.train_grpo import run_sft_warmup, run_grpo
from pathlib import Path

# SFT format priming (skip with SFT_WARMUP_STEPS=0)
if SFT_WARMUP_STEPS > 0:
    run_sft_warmup(model, tokenizer, ENV_URL, SFT_WARMUP_STEPS)

# GRPO training (uses adaptive curriculum from server/curriculum.py if importable)
output_dir = Path('/content/trained')
run_grpo(
    model, tokenizer, ENV_URL,
    max_steps=MAX_TRAINING_STEPS,
    num_generations=NUM_GENERATIONS,
    lr=LR,
    kl_coef=0.04,
    clip_eps=0.2,
    max_new_tokens=80,
    max_episode_steps=20,
    gen_temp=0.7,
    gen_top_p=0.95,
    save_every=100,
    audit_every=25,
    output_dir=output_dir,
    use_adaptive=True,
)

## 9. Save trained model to HF Hub (asks for HF token)

In [ ]:
# UNCOMMENT after training, and only after you confirm the run:
# from huggingface_hub import login
# login()  # paste token interactively
# model.push_to_hub('YOUR-USER/pharmarl-llama-trained')
# tokenizer.push_to_hub('YOUR-USER/pharmarl-llama-trained')

## 10. After-training smoke run (compare to baseline)

In [ ]:
FastLanguageModel.for_inference(model)

# (1) Trained measurement on HELD-OUT target — same setup as untrained baseline.
print(f'\nTrained model on HELD-OUT target {HELD_OUT_TARGET}:')
trained_held_out = [rollout_episode(model, tokenizer, ENV_URL, EVAL_DIFFICULTY, HELD_OUT_TARGET)
                    for _ in range(EVAL_EPISODES_PER_TARGET)]
trained_mean = statistics.mean(r['cumulative'] for r in trained_held_out)
print(f'  Trained mean cumulative on {HELD_OUT_TARGET}: {trained_mean:+.3f}')
for i, r in enumerate(trained_held_out):
    print(f'  ep{i+1}: cum={r["cumulative"]:+.3f}  final={r["final_smiles"]}')

# (2) Transfer summary — the headline result of the held-out generalization test.
delta = trained_mean - untrained_mean
print('\n' + '=' * 60)
print(f'  HELD-OUT TRANSFER TEST — target = {HELD_OUT_TARGET}')
print('=' * 60)
print(f'  Untrained model mean cumulative: {untrained_mean:+.3f}')
print(f'  Trained model mean cumulative:   {trained_mean:+.3f}')
print(f'  Delta (trained - untrained):     {delta:+.3f}')
print(f'  Trained / Untrained ratio:       {trained_mean / max(0.01, abs(untrained_mean)):.2f}x')
verdict = (
    'TRANSFER (trained > untrained by >50% gap)'
    if delta > 0 and (delta / max(0.01, abs(untrained_mean))) > 0.5
    else ('weak/no transfer (delta within noise)' if abs(delta) < 1.0 else 'partial transfer')
)
print(f'  Verdict: {verdict}')

## 11. Schema drift demo (Patronus AI sub-theme)

Demonstrates `schema_drift_enabled=True` — the env can switch reward weights mid-episode (early_admet vs late_potency profiles). Runs in-process, no HF Space required.

In [ ]:
# Schema drift demo — in-process env, no server.
import statistics, random as _random
from dataclasses import replace
from server.curriculum import DEFAULT_CONFIG
from server.drug_discovery_environment import DrugDiscoveryEnvironment
from models import MoleculeAction


def _scripted_rollout(env, max_steps: int = 12) -> float:
    """Drive the env with a simple ADD_FRAGMENT loop, then TERMINATE.

    Not a smart agent — the point is to compare reward STATISTICS across
    drift profiles, not to maximize reward.
    """
    cumulative = 0.0
    fragment_pool = ['C', 'O', 'N']
    for i in range(max_steps - 1):
        frag = fragment_pool[i % len(fragment_pool)]
        obs = env.step(MoleculeAction(action_type='ADD_FRAGMENT', fragment=frag, position=0))
        cumulative += obs.reward
        if obs.done:
            return cumulative
    obs = env.step(MoleculeAction(action_type='TERMINATE'))
    cumulative += obs.reward
    return cumulative


def schema_drift_rollout(profile: str, episodes: int = 20):
    cfg = replace(DEFAULT_CONFIG, schema_drift_enabled=True)
    env = DrugDiscoveryEnvironment(seed=0, config=cfg)
    cums = []
    for ep in range(episodes):
        env.reset(difficulty='hard', drift_profile=profile)
        cums.append(_scripted_rollout(env))
    mean = statistics.mean(cums)
    std = statistics.stdev(cums) if len(cums) > 1 else 0.0
    return mean, std


print('Schema drift rollouts (Patronus AI sub-theme):')
for profile in ('static', 'early_admet', 'late_potency'):
    mean, std = schema_drift_rollout(profile)
    print(f'  {profile:15s}: mean cumulative = {mean:+.3f} +/- {std:.3f}')

## 12. Multi-actor critic demo (Halluminate sub-theme)

`critic_enabled=True` makes the env populate `obs.metadata.critique` each step — a chemist/safety persona reviews the molecule and surfaces issues. Runs in-process.

In [ ]:
# Critic-agent demo — in-process env, no server required.
import random
from server.curriculum import CurriculumConfig
from server.drug_discovery_environment import DrugDiscoveryEnvironment
from models import MoleculeAction

config = CurriculumConfig(critic_enabled=True)
env = DrugDiscoveryEnvironment(seed=7, config=config)
obs = env.reset(difficulty='easy')
print(f'Start molecule: {obs.smiles}')
print('-' * 60)

rng = random.Random(7)
for step_i in range(10):
    fragment = rng.choice(obs.available_fragments)
    action = MoleculeAction(action_type='ADD_FRAGMENT', fragment=fragment, position=0)
    obs = env.step(action)

    crit = obs.metadata.get('critique', {})
    summary = crit.get('summary', '(no critique — action invalid)')
    issue_codes = [i['code'] for i in crit.get('issues', [])]
    print(f'step {step_i + 1}: ADD({fragment:>15s}) -> {obs.smiles}')
    print(f'         {summary}  codes={issue_codes}')
    if obs.done:
        break

# In production, the policy reads obs.metadata['critique']['summary'] in its prompt
# and decides whether to revise. With critic_enabled=False (default), this metadata
# key is absent.

## 13. Statistical CI eval — trained model vs all baselines

In [ ]:
# Statistical CI eval for the trained model.
# Saves JSON in the same shape as examples/eval_with_ci.py output, so the
# downstream plot script (examples/plot_results.py) just works.
import json, os, statistics

FastLanguageModel.for_inference(model)

EVAL_TARGETS = list(TRAINING_TARGETS) + [HELD_OUT_TARGET]  # DRD2, GSK3B, JNK3
EVAL_EPS = 10  # 10 episodes per target = tight CI on T4 budget

all_cums = []
results = {}
for target in EVAL_TARGETS:
    print(f'\n=== Trained model on {target} ===')
    cums = []
    finals = []
    for ep in range(EVAL_EPS):
        r = rollout_episode(model, tokenizer, ENV_URL, EVAL_DIFFICULTY, target)
        cums.append(r['cumulative'])
        finals.append(r['final_smiles'])
        print(f'  ep{ep+1}: cum={r["cumulative"]:+.3f} final={r["final_smiles"]}')
    mean = statistics.mean(cums)
    std = statistics.stdev(cums) if len(cums) > 1 else 0.0
    results[target] = {
        'episodes': cums,
        'final_smiles': finals,
        'mean': round(mean, 4),
        'std': round(std, 4),
        'n': len(cums),
        'min': round(min(cums), 4),
        'max': round(max(cums), 4),
    }
    all_cums.extend(cums)
    print(f'  -> mean={mean:+.3f}  std={std:.3f}')

overall_mean = statistics.mean(all_cums)
overall_std = statistics.stdev(all_cums) if len(all_cums) > 1 else 0.0

out = {
    'policy': 'trained_llama',
    'env_url': ENV_URL,
    'difficulty': EVAL_DIFFICULTY,
    'training_steps': MAX_TRAINING_STEPS,
    'training_targets': list(TRAINING_TARGETS),
    'held_out_target': HELD_OUT_TARGET,
    'model': MODEL_NAME,
    'results': results,
    'overall': {
        'mean': round(overall_mean, 4),
        'std': round(overall_std, 4),
        'n': len(all_cums),
    },
}

os.makedirs('runs', exist_ok=True)
with open('runs/trained_llama.json', 'w') as f:
    json.dump(out, f, indent=2)

print(f'\n*** OVERALL trained model: mean={overall_mean:+.3f} std={overall_std:.3f} ***')
print(f'\nSaved runs/trained_llama.json. Drop this + baseline JSONs into:')
print(f'  python -m examples.plot_results --inputs runs/random.json runs/scripted.json runs/trained_llama.json')

# Quick comparison print
print('\n=== Baseline comparison (from docs/baselines.md) ===')
print(f'  Random:        +2.30  (we just got: {overall_mean:+.3f})')
print(f'  Scripted:      +2.81')
print(f'  Llama 3B:      +1.67')
print(f'  Llama 8B:      +2.45')
print(f'  Llama 70B:     +1.19')
print(f'  Gemini Flash:  +1.81')
print(f'  Gemini Pro:    +3.68')

## 14. Fleet AI oversight demo (pure-LLM, sub-theme bonus)

`oversight_enabled=True` makes the env call a Gemini reviewer at TERMINATE — produces a strategy summary, risk flags, and risk level. Requires `GEMINI_API_KEY` in env.

In [ ]:
# Fleet AI oversight demo — runs in-process, single LLM call per episode at TERMINATE.
import os, json
from dataclasses import replace
from server.curriculum import DEFAULT_CONFIG
from server.drug_discovery_environment import DrugDiscoveryEnvironment
from models import MoleculeAction

# Make sure GEMINI_API_KEY is in the environment.
# In Colab: from google.colab import userdata; os.environ['GEMINI_API_KEY']=userdata.get('GEMINI_API_KEY')
assert os.environ.get('GEMINI_API_KEY'), 'Set GEMINI_API_KEY before running this cell'

cfg = replace(DEFAULT_CONFIG, oversight_enabled=True)
env = DrugDiscoveryEnvironment(seed=0, config=cfg)

# Run 3 episodes with a simple ADD-then-TERMINATE rollout — the point is the oversight reports.
for ep in range(3):
    obs = env.reset(difficulty='easy')
    print(f"\n=== Episode {ep+1} (target={obs.target}, scaffold={obs.smiles}) ===")
    for frag in ('c1ccccc1', 'N', 'OC'):
        if frag in obs.available_fragments:
            obs = env.step(MoleculeAction(action_type='ADD_FRAGMENT', fragment=frag, position=0))
    obs = env.step(MoleculeAction(action_type='TERMINATE'))
    print(f"Final molecule: {obs.smiles}")
    print(f"Final reward: {obs.reward:+.3f}")
    overs = obs.metadata.get('oversight')
    if overs:
        print(f"\n[OVERSIGHT — {overs.get('model_name', 'unknown model')}]")
        print(f"  strategy_summary: {overs['strategy_summary']}")
        print(f"  risk_flags:       {overs['risk_flags']}")
        print(f"  risk_level:       {overs['risk_level'].upper()}")
        print(f"  explanation:      {overs['explanation']}")
    else:
        print('  (no oversight report — check GEMINI_API_KEY or oversight_enabled flag)')